### Installation

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [2]:
%%capture
# Install latest transformers for Gemma 3N
!pip install --no-deps --upgrade timm # Only for Gemma 3N

In [3]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3n-E4B-it",
    dtype = None, # None for auto detection
    max_seq_length = 1024,
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # full finetuning
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.7.11: Fast Gemma3N patching. Transformers: 4.54.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3N does not support SDPA - switching to eager!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.72G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

...

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

In [4]:
from transformers import TextStreamer
# Helper function for inference
def do_gemma_3n_inference(messages, max_new_tokens = 128):
    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to("cuda"),
        max_new_tokens = max_new_tokens,
        temperature = 1.0, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

In [5]:
messages = [{
    "role": "user",
    "content": [{ "type" : "text",
                  "text" : "I feel groggy, what can I do to feel better?." }]
}]
do_gemma_3n_inference(messages)

Okay, feeling groggy is no fun! Here's a breakdown of things you can try, ranging from quick fixes to longer-term solutions. I'll categorize them for easier browsing.  I'll also include some questions to help narrow down the best approach for *you*.



**1. Quick Fixes (Immediate Relief - 5-15 minutes):**

*   **Hydrate:**  Dehydration is a common cause of grogginess. Drink a glass of water *right now*.
*   **Move Your Body:**  Even a short walk, some stretching, or a few jumping jacks


In [6]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

In [7]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

We get the first 2000 rows of the dataset

In [8]:
from datasets import load_dataset
dataset = load_dataset("jshargo/mts-medical-trainset2", split = "train[:2000]")

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

mts_medical_trainset.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1197 [00:00<?, ? examples/s]

In [9]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Unsloth: Standardizing formats (num_proc=12):   0%|          | 0/1197 [00:00<?, ? examples/s]

In [10]:
dataset[100]

{'conversations': [{'role': 'system',
   'content': "You are a medical doctor conducting a patient consultation. Here is the patient's background information:\n\nSection: PASTMEDICALHX\nPatient Information: Significant for moderate to severe aortic stenosis, urinary tract infection, hypertension, chronic kidney disease (although her creatinine is near normal).\n\nBased on this context, ask appropriate follow-up questions to gather more information before providing medical advice. Use the questioning patterns that doctors typically employ."},
  {'role': 'user', 'content': "Hello, I'm here for my appointment."},
  {'role': 'assistant',
   'content': 'Can you tell me what problems you have had in the past. Or what medical issues are currently going on?'},
  {'role': 'user', 'content': 'You mean my heart problem and things?'},
  {'role': 'assistant', 'content': 'Yes.'},
  {'role': 'user',
   'content': 'Okay, I have hypertension and kidney problems since many years.'},
  {'role': 'assistan

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`. We remove the `<bos>` token using removeprefix(`'<bos>'`) since we're finetuning. The Processor will add this token before training and the model expects only one.

In [11]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/1197 [00:00<?, ? examples/s]

Let's see how the chat template did! Notice there is no `<bos>` token as the processor tokenizer will be adding one.

In [12]:
dataset[100]["text"]

"<start_of_turn>user\nYou are a medical doctor conducting a patient consultation. Here is the patient's background information:\n\nSection: PASTMEDICALHX\nPatient Information: Significant for moderate to severe aortic stenosis, urinary tract infection, hypertension, chronic kidney disease (although her creatinine is near normal).\n\nBased on this context, ask appropriate follow-up questions to gather more information before providing medical advice. Use the questioning patterns that doctors typically employ.\n\nHello, I'm here for my appointment.<end_of_turn>\n<start_of_turn>model\nCan you tell me what problems you have had in the past. Or what medical issues are currently going on?<end_of_turn>\n<start_of_turn>user\nYou mean my heart problem and things?<end_of_turn>\n<start_of_turn>model\nYes.<end_of_turn>\n<start_of_turn>user\nOkay, I have hypertension and kidney problems since many years.<end_of_turn>\n<start_of_turn>model\nOkay.<end_of_turn>\n<start_of_turn>user\nI get frequent uri

<a name="Train"></a>
### Train the model
Huggingface TRL's `SFTTrainer`! [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer).

In [13]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1197 [00:00<?, ? examples/s]

use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [14]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=12):   0%|          | 0/1197 [00:00<?, ? examples/s]

verify masking the instruction part is done; only has a single `<bos>`

In [15]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<bos><start_of_turn>user\nYou are a medical doctor conducting a patient consultation. Here is the patient's background information:\n\nSection: PASTMEDICALHX\nPatient Information: Significant for moderate to severe aortic stenosis, urinary tract infection, hypertension, chronic kidney disease (although her creatinine is near normal).\n\nBased on this context, ask appropriate follow-up questions to gather more information before providing medical advice. Use the questioning patterns that doctors typically employ.\n\nHello, I'm here for my appointment.<end_of_turn>\n<start_of_turn>model\nCan you tell me what problems you have had in the past. Or what medical issues are currently going on?<end_of_turn>\n<start_of_turn>user\nYou mean my heart problem and things?<end_of_turn>\n<start_of_turn>model\nYes.<end_of_turn>\n<start_of_turn>user\nOkay, I have hypertension and kidney problems since many years.<end_of_turn>\n<start_of_turn>model\nOkay.<end_of_turn>\n<start_of_turn>user\nI get frequen

print the masked out example

In [16]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                        Can you tell me what problems you have had in the past. Or what medical issues are currently going on?<end_of_turn>\n                Yes.<end_of_turn>\n                    Okay.<end_of_turn>\n              Your urine creatinine is normal.<end_of_turn>\n              '

In [17]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.161 GB.
9.629 GB of memory reserved.


# Training the model

To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [18]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,197 | Num Epochs = 2 | Total steps = 600
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 19,210,240 of 7,869,188,432 (0.24% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,9.933600
2,12.137300
3,3.540200
4,4.076100
5,3.350900
6,3.420400
7,2.858800
8,2.792200
9,3.069000
10,2.879600


In [19]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

3548.7176 seconds used for training.
59.15 minutes used for training.
Peak reserved memory = 11.652 GB.
Peak reserved memory for training = 2.023 GB.
Peak reserved memory % of max memory = 52.579 %.
Peak reserved memory for training % of max memory = 9.129 %.


In [20]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "Continue the sequence: 1, 1, 2, 3, 5, 8,",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

['<bos><start_of_turn>user\nContinue the sequence: 1, 1, 2, 3, 5, 8,<end_of_turn>\n<start_of_turn>model\n13, 21, 34...<end_of_turn>']

In [21]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Why is the sky blue?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

The sky is blue due to a phenomenon called **Rayleigh scattering**. Here's a breakdown of how it works:

* **Sunlight and Colors:** Sunlight is actually made up of all the colors of the rainbow.
* **Entering the Atmosphere:** When sunlight enters the Earth's atmosphere, it collides


<a name="Save"></a>
### Saving, loading finetuned models


This only saves the LoRA adapters, and not the full model.

In [22]:
model.push_to_hub("jshargo/gemma-3-LoRA-4B", token = "HF_ACCESS_TOKEN")
tokenizer.push_to_hub("jshargo/gemma-3-LoRA-4B", token = "HF_ACCESS_TOKEN") # Online saving

README.md:   0%|          | 0.00/604 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/76.9M [00:00<?, ?B/s]

Saved model to https://huggingface.co/jshargo/gemma-3-LoRA-4B


No files have been modified since last commit. Skipping to prevent empty commit.


In [24]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
from google.colab import userdata
import torch

if True:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "jshargo/gemma-3-LoRA-4B", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 1024,
        load_in_4bit = True,
        token=userdata.get('HF_ACCESS_TOKEN'),
    )

==((====))==  Unsloth 2025.7.11: Fast Gemma3N patching. Transformers: 4.54.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/76.9M [00:00<?, ?B/s]

### Saving to float16 for VLLM

In [25]:
if True: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3N-finetune-4B", tokenizer)

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Merging weights into 16bit:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit:  25%|██▌       | 1/4 [00:19<00:57, 19.22s/it]

...

Unsloth: Merging weights into 16bit:  50%|█████     | 2/4 [00:51<00:54, 27.02s/it]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [01:31<00:32, 32.69s/it]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:48<00:00, 27.22s/it]


In [26]:
if True: # Change to True to upload finetune
    model.push_to_hub_merged(
        "jshargo/gemma-3N-finetune-4B", tokenizer,
        token = "HF_ACCESS_TOKEN",
    )

  0%|          | 0/2 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [04:45<01:41, 101.38s/it]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:43<00:00, 86.00s/it]
